<a href="https://colab.research.google.com/github/parika8ec-hub/DA_AI_Project3_Stock_Data/blob/main/Project3_MA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Note: For Machine Learning for Predicting Trading Signals Tasks, Cleaned data of project 2 used.

In [25]:
#Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

In [7]:
#Loading Cleaned dataset of Project2
# and low_memory=False prevents dtype warning by reading the file in one go and this also helps pandas correctly infer
# data types for columns with mixed values
stock_clean_data = pd.read_csv('cleaned_stock_data.csv', low_memory=False)
#Display few rows of dataset
print('Few rows of Cleaned Data:',stock_clean_data.head())


Few rows of Cleaned Data:          date  ticker      open     close  adj_close       low      high  \
0  1972-07-13     148 -1.137557 -1.139619   0.000666 -1.135742 -1.141196   
1  1972-07-14     148 -1.138895 -1.138280   0.000681 -1.135573 -1.140866   
2  1972-07-17     148 -1.137557 -1.138782   0.000675 -1.135573 -1.140866   
3  1972-07-18     148 -1.138226 -1.139452   0.000667 -1.136927 -1.141858   
4  1972-07-19     148 -1.138728 -1.139117   0.000671 -1.135742 -1.141527   

     volume  price_decade  close_ma7  ...  sector_CONSUMER NON-DURABLES  \
0  1.610612        1970.0   0.924479  ...                         False   
1  2.052429        1970.0   0.928943  ...                         False   
2  0.980964        1970.0   0.929688  ...                         False   
3  2.052429        1970.0   0.927083  ...                         False   
4  2.052429        1970.0   0.924851  ...                         False   

   sector_CONSUMER SERVICES  sector_ENERGY  sector_FINANCE  \
0   

In [8]:
#Display information of data
print('Information of Data:',stock_clean_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 624544 entries, 0 to 624543
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   date                          624544 non-null  object 
 1   ticker                        624544 non-null  int64  
 2   open                          624544 non-null  float64
 3   close                         624544 non-null  float64
 4   adj_close                     624544 non-null  float64
 5   low                           624544 non-null  float64
 6   high                          624544 non-null  float64
 7   volume                        624544 non-null  float64
 8   price_decade                  624544 non-null  float64
 9   close_ma7                     624544 non-null  float64
 10  close_ma30                    624544 non-null  float64
 11  close_volatility              624544 non-null  float64
 12  daily_return                  624544 non-nul

In [9]:
# Convert date column to datetime
stock_clean_data['date'] = pd.to_datetime(stock_clean_data['date'])

# Sort values by ticker and date
stock_clean_data = stock_clean_data.sort_values(by=['ticker', 'date'])

# Task-1: Feature Engineering with Technical Indicators

**i) Compute MACD (Moving Average Convergence Divergence ) Calculation**

In [10]:
# Create EMA_12 column by calculating 12-day Exponential Moving Average (EMA) of closing prices groupby('ticker'),which
# ensures calculation is done separately for each stock and ewm(span=12) gives more weight to recent prices over the last 12 periods
stock_clean_data['EMA_12'] = stock_clean_data.groupby('ticker')['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())

# Craete EMA_26 column by calculating 26-day Exponential Moving Average (EMA) of closing prices and this represents the
#longer-term price trend
stock_clean_data['EMA_26'] = stock_clean_data.groupby('ticker')['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())

# Create MACD column by calculating MACD line by using formula as MACD = EMA_12 - EMA_26
# and Positive MACD suggests upward momentum and Negative MACD suggests downward momentum
stock_clean_data['MACD'] = stock_clean_data['EMA_12'] - stock_clean_data['EMA_26']

# Create MACD_signal column by calculating Signal Line (9-day EMA of MACD), which is the 9-day EMA of the MACD values
# and it helps identify buy/sell crossover signals
stock_clean_data['MACD_signal'] = stock_clean_data.groupby('ticker')['MACD'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

# Create MACD_trade_signal column by identifying actual MACD crossovers
stock_clean_data['MACD_trade_signal'] = np.where(

    # Buy Signal=Current MACD is above current signal line and previous MACD was below or equal to previous signal line,
    # which confirms MACD has crossed above the signal line
    (stock_clean_data['MACD'] > stock_clean_data['MACD_signal']) &
    (stock_clean_data['MACD'].shift(1) <= stock_clean_data['MACD_signal'].shift(1)),
    'Buy',
    np.where(
        # Sell Signal= Current MACD is below current signal line and previous MACD was above or equal to previous signal line,
        # Which confirms MACD has crossed below the signal line
        (stock_clean_data['MACD'] < stock_clean_data['MACD_signal']) &
        (stock_clean_data['MACD'].shift(1) >= stock_clean_data['MACD_signal'].shift(1)),
        'Sell',

        # If no crossover occurs, assign Hold signal
        'Hold'))


#Verify adding of new columns in dataset by display few rows of dataset
print('Few rows in dataset:',stock_clean_data.head())

Few rows in dataset:            date  ticker      open     close  adj_close       low      high  \
6227 1981-01-27       0 -1.160446 -1.161477   0.025807 -1.157514 -1.163823   
6251 1981-01-29       0 -1.163026 -1.163916   0.024094 -1.159982 -1.166374   
6267 1981-01-30       0 -1.164747 -1.165782   0.022783 -1.161868 -1.168075   
6274 1981-02-02       0 -1.166754 -1.167647   0.021473 -1.163755 -1.170058   
6291 1981-02-03       0 -1.165750 -1.166499   0.022279 -1.162594 -1.168925   

        volume  price_decade  close_ma7  ...  sector_MISCELLANEOUS  \
6227  2.052429        1980.0   0.576212  ...                 False   
6251  2.052429        1980.0   0.573342  ...                 False   
6267  2.052429        1980.0   0.561543  ...                 False   
6274  2.052429        1980.0   0.548151  ...                 False   
6291  2.052429        1980.0   0.535714  ...                 False   

      sector_PUBLIC UTILITIES  sector_TECHNOLOGY  sector_TRANSPORTATION  \
6227          

**ii) Compute RSI (Relative Strength Index) Calculation**

In [11]:
# Create price_change column by calculating day-to-day change in closing price for each stock separately using groupby('ticker'),
# which ensures each stock is processed independently and diff() subtracts current close price from previous day's close price
stock_clean_data['price_change'] = stock_clean_data.groupby('ticker')['close'].diff()

# Create gain column by computing gain as if price increased (positive change), keep that value Otherwise assign 0
stock_clean_data['gain'] = np.where(stock_clean_data['price_change'] > 0,stock_clean_data['price_change'],0)

# Create loss column by computing loss as if price decreased (negative change), store absolute value of loss otherwise assign 0
stock_clean_data['loss'] = np.where(stock_clean_data['price_change'] < 0, abs(stock_clean_data['price_change']), 0)

# Create avg_gain column by calculating 14-period Exponential Moving Average(EMA) of gains, which  gives more weight to recent gains
# This used in RSI calculation Average gain (14-day EMA)
stock_clean_data['avg_gain'] =stock_clean_data.groupby('ticker')['gain'].transform(lambda x: x.ewm(span=14, adjust=False).mean())

# Create avg_loss column by calculating 14-period Exponential Moving Average(EMA) of losses,which gives more weight to recent losses
# This used in RSI calculation Average loss (14-day EMA)
stock_clean_data['avg_loss'] = stock_clean_data.groupby('ticker')['loss'].transform(lambda x: x.ewm(span=14, adjust=False).mean())

# Create RS column by calculating Relative Strength (RS) using formula as EMA(Average Gain) / EMA(Average Loss)
stock_clean_data['RS'] = stock_clean_data['avg_gain'] /stock_clean_data['avg_loss']

# Create RSI column by calculating Relative Strength Index (RSI) using formula as RSI = 100 - (100 / (1 + RS)) where RSI values range from 0 to 100
stock_clean_data['RSI'] = 100 - (100 / (1 + stock_clean_data['RS']))

# Create RSI_signal column as if RSI below 30 (oversold condition) means Buy, RSI above 70 (overbought condition) means Sell and
# RSI between 30 and 70 means Hold
stock_clean_data['RSI_signal'] = np.where(stock_clean_data['RSI'] < 30,'Buy',np.where(stock_clean_data['RSI'] > 70,'Sell','Hold'))

#Verify adding of new columns in dataset by display few rows of dataset
print('Few rows in dataset:',stock_clean_data.head())

Few rows in dataset:            date  ticker      open     close  adj_close       low      high  \
6227 1981-01-27       0 -1.160446 -1.161477   0.025807 -1.157514 -1.163823   
6251 1981-01-29       0 -1.163026 -1.163916   0.024094 -1.159982 -1.166374   
6267 1981-01-30       0 -1.164747 -1.165782   0.022783 -1.161868 -1.168075   
6274 1981-02-02       0 -1.166754 -1.167647   0.021473 -1.163755 -1.170058   
6291 1981-02-03       0 -1.165750 -1.166499   0.022279 -1.162594 -1.168925   

        volume  price_decade  close_ma7  ...  MACD_signal  MACD_trade_signal  \
6227  2.052429        1980.0   0.576212  ...     0.000000               Hold   
6251  2.052429        1980.0   0.573342  ...    -0.000039               Sell   
6267  2.052429        1980.0   0.561543  ...    -0.000130               Hold   
6274  2.052429        1980.0   0.548151  ...    -0.000278               Hold   
6291  2.052429        1980.0   0.535714  ...    -0.000436               Hold   

      price_change      gain 

**iii) Display Final Combined Signal**

In [12]:
# Create Final_signal column as final trading signal by combining MACD and RSI signals
stock_clean_data['Final_signal'] = np.where(
     # If both MACD and RSI indicate Buy, Final signal = Buy
    (stock_clean_data['MACD_trade_signal'] == 'Buy') & (stock_clean_data['RSI_signal'] == 'Buy'), 'Buy',
     # Otherwise check if both MACD and RSI indicate Sell, Final signal = Sell
    np.where((stock_clean_data['MACD_trade_signal'] == 'Sell') & (stock_clean_data['RSI_signal'] == 'Sell'),'Sell',
               # If neither condition is met, Final signal = Hold
             'Hold'))

# Display few rows of dataset to verify signal columns
print(stock_clean_data[['date','ticker','close','MACD','MACD_signal','RSI','MACD_trade_signal','RSI_signal','Final_signal']].head())

           date  ticker     close      MACD  MACD_signal        RSI  \
6227 1981-01-27       0 -1.161477  0.000000     0.000000        NaN   
6251 1981-01-29       0 -1.163916 -0.000195    -0.000039   0.000000   
6267 1981-01-30       0 -1.165782 -0.000494    -0.000130   0.000000   
6274 1981-02-02       0 -1.167647 -0.000871    -0.000278   0.000000   
6291 1981-02-03       0 -1.166499 -0.001065    -0.000436  19.951368   

     MACD_trade_signal RSI_signal Final_signal  
6227              Hold       Hold         Hold  
6251              Sell        Buy         Hold  
6267              Hold        Buy         Hold  
6274              Hold        Buy         Hold  
6291              Hold        Buy         Hold  


# Task-2: Data Preparation and Splitting


In [13]:
# Select important features for modeling
features = ['open', 'close', 'high', 'low', 'volume','MACD', 'MACD_signal', 'RSI']

#Feature selection
X = stock_clean_data[features] #Select Input features (X)
y = stock_clean_data['Final_signal'] #Select Target variable (y) as Final_signal,which contains Buy, Sell, Hold labels

# Remove rows with missing values because missing values may exist due to EMA/RSI calculations at the beginning
data_final = pd.concat([X, y], axis=1).dropna()

# Separate features and target again after removing null values
X = data_final[features]
y = data_final['Final_signal']


# Split dataset into 80% training and 20% testing sets without shuffling
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=False)

# Display dataset shapes
print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)
print("Training Labels Shape:", y_train.shape)
print("Testing Labels Shape:", y_test.shape)

Training Features Shape: (499441, 8)
Testing Features Shape: (124861, 8)
Training Labels Shape: (499441,)
Testing Labels Shape: (124861,)


# Task-3: Model Building and Validation

**i) Logistic Regression Model Building:**

In [14]:
# Initialize model
log_model = LogisticRegression(max_iter=1000)

# Train model on training data
log_model.fit(X_train, y_train)

# Predictions on test data
y_pred_lr = log_model.predict(X_test)

**ii) Random Forest Classifier Model Building:**

In [15]:
# Initialize model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model on training data
rf_model.fit(X_train, y_train)

# Predictions on test data
y_pred_rf = rf_model.predict(X_test)

**iii) Support Vector Machine (SVM) Model Building:**

In [16]:
# Initialize model
svm_model = SVC(kernel='rbf')

# Train model on training data
svm_model.fit(X_train, y_train)

# Predictions on test data
y_pred_svm = svm_model.predict(X_test)

# Task-4: Model Evaluation and Optimization

**i) Model Evaluation:**

In [20]:
# Evaluation using accuracy_score,confusion matrix, classification report and cross validation accuracy
print('-'*45)
print('Logistic Regression Model Evaluation:')
print('-'*45)
print('\nAccuracy:', round(accuracy_score(y_test, y_pred_lr),3))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr,zero_division=0))
cv_lr = cross_val_score(log_model, X, y, cv=5, scoring='accuracy')
print('\nCross Validation Accuracy:', round(cv_lr.mean(),3))

# Evaluation using accuracy_score,confusion matrix, classification report and cross validation accuracy
print('\n\n')
print('-'*45)
print('Random Forest Model Evaluation:')
print('-'*45)
print('\nAccuracy:', round(accuracy_score(y_test, y_pred_rf),3))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_rf))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf,zero_division=0))
cv_rf = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')
print('\nCross Validation Accuracy:', round(cv_rf.mean(),3))

# Evaluation using accuracy_score,confusion matrix, classification report and cross validation accuracy
print('\n\n')
print('-'*45)
print('SVM Model Evaluation:')
print('-'*45)
print('\nAccuracy:', round(accuracy_score(y_test, y_pred_svm),3))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_svm))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_svm,zero_division=0))
cv_svm = cross_val_score(svm_model, X, y, cv=5, scoring='accuracy')
print('\nCross Validation Accuracy:', round(cv_svm.mean(),3))

---------------------------------------------
Logistic Regression Model Evaluation:
---------------------------------------------

Accuracy: 0.999

Confusion Matrix:
[[     0     30      0]
 [     0 124793      0]
 [     0     38      0]]

Classification Report:
              precision    recall  f1-score   support

         Buy       0.00      0.00      0.00        30
        Hold       1.00      1.00      1.00    124793
        Sell       0.00      0.00      0.00        38

    accuracy                           1.00    124861
   macro avg       0.33      0.33      0.33    124861
weighted avg       1.00      1.00      1.00    124861


Cross Validation Accuracy: 0.999



---------------------------------------------
Random Forest Model Evaluation:
---------------------------------------------

Accuracy: 0.999

Confusion Matrix:
[[     0     30      0]
 [     0 124793      0]
 [     0     38      0]]

Classification Report:
              precision    recall  f1-score   support

       

**Interpretation of All Models Result:**

- Although Logistic Regression, Random Forest and SVM all achieved a very high accuracy of approximately 0.999, the confusion matrices show that the models are not truly performing well across all classes.
- The predictions are heavily biased toward the “Hold” class, which dominates the dataset, while the minority classes “Buy” and “Sell” are not being correctly identified at all, resulting in zero precision and recall for these classes.
- This indicates a strong class imbalance issue in the dataset, where the model learns to predict the majority class to achieve high accuracy rather than learning meaningful patterns for all trading signals.
- Therefore, despite the high accuracy, the model performance is misleading, and metrics such as precision, recall and F1-score clearly show that the model is not effective in distinguishing Buy and Sell signals.

**ii) Hyperparameter Optimization:**

**Note for using StandardScaler:** Feature scaling was applied only for the SVM model because SVM is sensitive to the scale of input features and relies on distance-based calculations to determine decision boundaries. In this stock dataset, features such as volume, RSI, MACD, and price variables exist on different numerical scales, which could negatively affect SVM performance if left unscaled. Therefore, StandardScaler was used to normalize the training and testing data before training and tuning the SVM model. Scaling was not required for Random Forest, and Logistic Regression was trained on the original feature values.

**Note for reducing sample size for SVM:**

Due to the large size of the dataset (~500,000 records), SVM hyperparameter tuning using GridSearchCV on the full dataset was computationally expensive and time-consuming. Since Support Vector Machines have high computational complexity and do not scale efficiently with very large datasets, a representative random sample of 100,000 records was used for model training and hyperparameter optimization. This approach reduces execution time while still preserving the underlying data distribution, ensuring that the model learns meaningful patterns from the stock market data without significant loss of generalization performance.

In [33]:

#Take 100K sample from training data
X_sample = X_train.sample(n=100000, random_state=42)
y_sample = y_train.loc[X_sample.index]

# Initialize scaler
scaler = StandardScaler()

# Fit scaler on training data and transform training data
X_train_scaled = scaler.fit_transform(X_sample)

# Use same scaler to transform testing data
X_test_scaled = scaler.transform(X_test)

In [22]:
#Logistic Regression Tuning--------------------------------------------
# Define hyperparameter grid for Logistic Regression as C=inverse of regularization strength as smaller values = stronger regularization
# and solver as optimization algorithm used to find best parameters
param_grid_lr = {'C': [0.01, 0.1, 1, 10],'solver': ['lbfgs', 'liblinear']}

# Initialize GridSearchCV for Logistic Regression tuning as LogisticRegression(max_iter=1000) ensures convergence during training,
# cv=3 as 3-fold cross-validation for reliable evaluation and scoring=accuracy as evaluation metric
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000),param_grid_lr,cv=3,scoring='accuracy')

# Train GridSearchCV on training data, which tries all combinations of hyperparameters and selects the best model
grid_lr.fit(X_train, y_train)

print("Best LR Parameters:", grid_lr.best_params_)#display best hyperparameters found by GridSearchCV during tuning
print("Best LR CV Score:", grid_lr.best_score_)#display best cross-validation accuracy score achieved

Best LR Parameters: {'C': 0.01, 'solver': 'lbfgs'}
Best LR CV Score: 0.9993312523481288


In [23]:
#Random Forest Tuning--------------------------------------------------
# Define the hyperparameter grid for Random Forest as n_estimators=number of trees in the forest
# ,max_depth= maximum depth of each tree (controls overfitting) and min_samples_split= minimum samples required to split a node
param_grid_rf = {'n_estimators': [50, 100, 200],'max_depth': [10, 20, None],'min_samples_split': [2, 5]}

# Initialize GridSearchCV for Random Forest optimization as RandomForestClassifier(random_state=42) ensures reproducible results
# , cv=3 as 3-fold cross-validation for model validation, scoring='accuracy' as evaluation metric
# and n_jobs=-1 as uses all CPU cores for faster computation
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),param_grid_rf,cv=3,scoring='accuracy',n_jobs=-1)

# Train GridSearchCV on training data, which tries all combinations of hyperparameters and selects the best model
grid_rf.fit(X_train, y_train)

print("Best RF Parameters:", grid_rf.best_params_)#display best hyperparameters found by GridSearchCV during tuning
print("Best RF CV Score:", grid_rf.best_score_)#display best cross-validation accuracy score achieved

Best RF Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}
Best RF CV Score: 0.9993312523481288


In [35]:
#SVM Hyperparameter Tuning---------------------------------------------
# Define hyperparameter grid for SVM as C=controls regularization strength as lower values = more regularization,
#kernel=type of decision boundary as linear because for 50K data, non linear kernel will take more time to run
param_grid_svm = {'C': [0.1, 1, 10],'kernel': ['linear']}

# Initialize GridSearchCV for SVM model optimization as SVC()=Support Vector Classifier, cv=3 as 3-fold cross-validation
# ,scoring=accuracy as evaluation metric and n_jobs=-1 as uses all CPU cores for faster execution
grid_svm = GridSearchCV(SVC(),param_grid_svm,cv=3,scoring='accuracy',n_jobs=-1)

# Train GridSearchCV on training data, which tries all combinations of hyperparameters and selects the best model
grid_svm.fit(X_train_scaled, y_sample)

print("Best SVM Parameters:", grid_svm.best_params_)#display best hyperparameters found by GridSearchCV during tuning
print("Best SVM CV Score:", grid_svm.best_score_)#display best cross-validation accuracy score achieved

Best SVM Parameters: {'C': 0.1, 'kernel': 'linear'}
Best SVM CV Score: 0.999420000199882


**Interpretation of Hyperparameter Optimization Result:**

- The hyperparameter tuning results show that all three models Logistic Regression, Random Forest and SVM achieved extremely high and very similar cross-validation scores, all close to 0.999.

- For Logistic Regression, the best configuration was a very small regularization value (C = 0.01) with the 'lbfgs' solver, indicating that a strongly regularized linear model was sufficient to fit the data.

- Random Forest performed best with a relatively shallow tree structure (max_depth = 10) and a small number of trees (n_estimators = 50), suggesting that the patterns in the dataset are easy to capture and do not require complex ensembles.

- SVM achieved the highest score among the three models using a linear kernel with C = 0.1, showing that a simple linear decision boundary is enough to separate the classes effectively.

- However, despite these near-perfect cross-validation scores, the earlier evaluation results show that the models are heavily biased toward predicting the majority class (“Hold”) and fail to correctly identify the minority classes (“Buy” and “Sell”).

- This indicates that the high accuracy is largely influenced by class imbalance rather than true predictive power.

- Therefore, while SVM slightly outperforms the other models in CV score, all models demonstrate similar behavior and limited effectiveness in detecting meaningful trading signals.

**Model Comparision:**

- SVM performed slightly better than the other models, with the highest CV score (~0.99942), suggesting a linear decision boundary is sufficient for this dataset.
- Logistic Regression and Random Forest performed nearly the same, showing no significant performance difference between linear and tree-based approaches.
- Logistic Regression selected strong regularization (C = 0.01), indicating that a simple linear model is sufficient.
- Random Forest selected a shallow tree structure (max_depth = 10, n_estimators = 50), suggesting the data does not require complex trees to fit patterns.
- Despite high accuracy, all models show similar behavior in predictions, meaning no model clearly outperforms the others in practical terms.
- All models are biased toward predicting the majority class, making accuracy an unreliable metric for evaluation.